In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
import rasterio
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.windows import from_bounds
from matplotlib.patches import Patch
import pyreadr
from shapely.geometry import Point
from datetime import datetime, timedelta
import seaborn as sns

pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
# data load 
epa_stations = gpd.read_file('../01_data/01_raw/childs_pm/epa_station_locations/epa_station_locations.shp')
station_smoke = pyreadr.read_r('../01_data/01_raw/childs_pm/station_smokePM_2025_01.rds')[None]
counties = gpd.read_file('~/Desktop/Desktop/epidemiology_PhD/01_data/clean/us_cnty_boundaries.geojson')

In [ ]:
epa_stations.explore()

In [ ]:
epa_stations.head()

# grid_5km: 5km grid they constructed, they assigned each monitor to each grid cell and then defined the smoke for each epa station that way.
# shouldn't matter here.
# these are the 5km grid cell IDs. 

In [ ]:
station_smoke
station_smoke[station_smoke['id'] == '060371103'].head(25)

# smoke_day: whether there was a smoke plume intersecting the 5km grid cell for that station on that day.
# LA_wildfire_day: is it plausible that that grid cell is affected by the LA wildfires. this var is trying to determine whether the smoke that day at that station was due to LA wildfires. maybe just use this variable as a stratifier to look at, but prob don't have to use it in the analysis. just nice to know who is exposed to what.
# pm25_med_3yr: median pm2.5 on non-smoke days amongst all days for that station in that month and the 2 years prior.
    # pm25 - pm25_med_3yr = pm25_anom because they're looking at the anomalous smoke above the 3 yr median. and then the smoekPM var indicates whether it was a smoke day.
    # when smoke_day is 1, when there is a pm25_anom, that is attributed to smokePM and thus is the value in smokePM.
    # smokePM is noisy but not necessarily wrong. just may include some other anomalous PM but is close.
# smokePM: what is the PM2.5 that we think is from wf smoke? 
# light/med/dense is a measure of smoke.

# probably use the smokePM variable bc thats the smoke pm. it will be 0 when it was not a smoke day.

# NOTE: there are no stations in the palisades fire area. since the wind blew toward the water, there was just nothing to pick up on smoke in that area bc all stations are behind it. not a ton of people affected by the palisades fire smoke, many more from eaton. we corroborated this with the modis satellite data and where there are missing data from the satellite, there just aren't a lot of people

In [ ]:
# combining the data 
station_smoke_gdf = epa_stations.merge(station_smoke, left_on='stn_id', right_on='id', how='left')
station_smoke_gdf = gpd.GeoDataFrame(station_smoke_gdf, crs=station_smoke_gdf.crs)
station_smoke_gdf.explore()

In [ ]:
# make some plots: 
# PLOTS: plot the stations with their values and take a look. should make it clear where the cone would go. also is the station under a smoke plume. and put the fire on those plots. and then we can define where the exposure is and where the cone should be. and then we can decide how to define the smoke exposure over the week period based on what we think is happening. 
# 	- do the first 4 days
# 	- make a time series line plot of station pm values and color them by region and have 6 groupings of monitors in space so we can see how the colors track together. 
# 	- make this + the map described above 

In [ ]:
st_smoke_gdf_sm = station_smoke_gdf[['stn_id', 'geometry', 'smokePM', 'smoke_day', 'date']]
st_smoke_gdf_sm_mercator = st_smoke_gdf_sm.to_crs(epsg=3857)
st_smoke_gdf_sm['date'] = pd.to_datetime(st_smoke_gdf_sm['date'])

# subset to the right geo area and the right dates
lat_north = 35.3  # South of Bakersfield
lat_south = 32.7  # North of San Diego  
lon_west = -119.5  # Western boundary
lon_east = -116.0  # Eastern boundary
st_smoke_gdf_sm['lon'] = st_smoke_gdf_sm['geometry'].apply(lambda x: x.x)
st_smoke_gdf_sm['lat'] = st_smoke_gdf_sm['geometry'].apply(lambda x: x.y)
socal_data = st_smoke_gdf_sm[
    (st_smoke_gdf_sm['lat'] >= lat_south) & 
    (st_smoke_gdf_sm['lat'] <= lat_north) &
    (st_smoke_gdf_sm['lon'] >= lon_west) & 
    (st_smoke_gdf_sm['lon'] <= lon_east)
]

la_fires_start = pd.to_datetime('2025-01-07')
first_week = [
    la_fires_start,
    la_fires_start + pd.Timedelta(days=1),  # Jan 8
    la_fires_start + pd.Timedelta(days=2),  # Jan 9
    la_fires_start + pd.Timedelta(days=3),  # Jan 10
    la_fires_start + pd.Timedelta(days=4),  # Jan 11
    la_fires_start + pd.Timedelta(days=5),  # Jan 12
    la_fires_start + pd.Timedelta(days=6)   # Jan 13
]


In [ ]:
# fig
fig, axes = plt.subplots(1, 7, figsize=(20, 5))
axes = axes.flatten()

# calc color range from socal data only
q01 = socal_data['smokePM'].quantile(0.01)
q99 = socal_data['smokePM'].quantile(0.99)
vmin = max(0, q01) 
vmax = q99

print(f"Color scale range: {vmin:.2f} to {vmax:.2f} µg/m³")

for i, day in enumerate(first_week):
    ax = axes[i]
    
    day_data = socal_data[socal_data['date'] == day]
    
    if len(day_data) == 0:
        ax.text(0.5, 0.5, 'No data\navailable', transform=ax.transAxes, 
                ha='center', va='center', fontsize=12)
        ax.set_title(f'{day.strftime("%m/%d")}', fontsize=12, fontweight='bold')
        ax.axis('off')
        continue
    
    day_data_mercator = day_data.to_crs(epsg=3857)
    
    # split data based on smoke_day values so we can differentiate markers
    
    smoke_day_data = day_data_mercator[day_data_mercator['smoke_day'].notna()]
    no_smoke_day_data = day_data_mercator[day_data_mercator['smoke_day'].isna()]
    
    # non-smoke day data = circles (default marker)
    if len(no_smoke_day_data) > 0:
        no_smoke_day_data.plot(
            column='smokePM', 
            cmap='viridis', 
            markersize=80, 
            alpha=0.9,
            edgecolor='white',
            linewidth=1,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            marker='s'  # squares for no smoke day
        )
    
    # smoke day data = squares
    if len(smoke_day_data) > 0:
        smoke_day_data.plot(
            column='smokePM', 
            cmap='viridis', 
            markersize=80, 
            alpha=0.9,
            edgecolor='white',
            linewidth=1,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            marker='o'  # circles for smoke day
        )

    try:
        ctx.add_basemap(ax, 
                        crs=day_data_mercator.crs.to_string(), 
                        source=ctx.providers.OpenStreetMap.Mapnik,
                        alpha=0.6)
    except Exception as e:
        print(f"Basemap error for day {i+1}: {e}")
    
    ax.set_title(f'{day.strftime("%m/%d")}', fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # little box with stats
    smoke_count = len(smoke_day_data)
    no_smoke_count = len(no_smoke_day_data)
    day_stats = f'n={len(day_data)}\n□={no_smoke_count} ○={smoke_count}\nmax pm={day_data["smokePM"].max():.0f}'
    ax.text(0.02, 0.02, day_stats, transform=ax.transAxes, 
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8),
            verticalalignment='bottom', fontsize=8)

# legend for marker shapes
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=8, label='No smoke day'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='Smoke day')
]
fig.legend(handles=legend_elements, loc='lower right')

# horizontal colorbar at the bottom
fig.subplots_adjust(bottom=0.15, top=0.85)
cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.06])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Smoke PM (µg/m³)', fontsize=12)

plt.suptitle('LA Fires smoke PM$_{2.5}$ January 7-13, 2025', 
             fontsize=14, fontweight='bold', y=0.92)

plt.tight_layout()
plt.show()

print("\n=== SUMMARY ===")
for i, day in enumerate(first_week):
    day_data = socal_data[socal_data['date'] == day]
    print(f"Day {i+1} ({day.strftime('%Y-%m-%d %A')}): {len(day_data)} stations with non-NA smoke PM included")
    if len(day_data) > 0:
        print(f"  PM range: {day_data['smokePM'].min():.1f}-{day_data['smokePM'].max():.1f} µg/m³")
        print(f"  Mean PM: {day_data['smokePM'].mean():.1f} µg/m³")
        high_pm_count = (day_data['smokePM'] > 10).sum()
        print(f"  Stations with PM > 10: {high_pm_count} ({high_pm_count/len(day_data)*100:.1f}%)")
    else:
        print("  No data available")
    print()

## DO THIS SAME THING BUT JUST FOR LA COUNTY SO WE CAN SEE THE SUMMARY STATS FOR JUST LA COUNTY. INTERSECT THE STATIONS WITH LA COUNTY TO DO THIS.

In [ ]:
# why do some days have fewer stations? IT is because some stations don't have data on some days. 

# let's look at day 3 station ID 06111004
stn_id = '061111004'
day2_data = socal_data[(socal_data['date'] == first_week[1]) & (socal_data['stn_id'] == stn_id)]
day2_data
day3_data = socal_data[(socal_data['date'] == first_week[2]) & (socal_data['stn_id'] == stn_id)]
day3_data

# for example, station id 061111004 is missing from day 3 but not other days. 

# some missingness is expected.

In [ ]:
# make a time series line plot of station pm values

# filter to first week and remove NAs
first_week_data = socal_data[socal_data['date'].isin(first_week)].copy()
first_week_data = first_week_data.dropna(subset=['smokePM'])

plt.figure(figsize=(12, 8))
unique_stations = first_week_data['stn_id'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_stations)))

# plot each station as a separate line using colors from the colormap
for i, station in enumerate(unique_stations):
    station_data = first_week_data[first_week_data['stn_id'] == station]
    
    # sort by date to ensure proper line connections
    station_data = station_data.sort_values('date')
    
    plt.plot(station_data['date'], station_data['smokePM'], 
             marker='o', linewidth=2, markersize=6,
             label=f'Station {station}', color=colors[i])

plt.title('PM2.5 levels by station - January 7-13, 2025', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('PM2.5 (μg/m³)', fontsize=12, fontweight='bold')

plt.xticks(first_week, [date.strftime('%b %d') for date in first_week], rotation=45)

# add legend and grid
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.ylim(bottom=0)
plt.show()


In [ ]:
# Create groups based on location 

# group 1: LA county (06037)
# group 2: ventura (06111), santa barbara (06083), san louis obispo (06079), kern (06029)
# group 3: orange (06059), san diego (06073), imperial (06025), riverside (06065), san bernadino (06071)

# define county groups
county_groups = {
    # group 1: LA County
    '06037': 'LA',
    
    # group 2: northwest counties
    '06111': 'Northwest',  # Ventura
    '06083': 'Northwest',  # Santa Barbara
    '06079': 'Northwest',  # San Luis Obispo
    '06029': 'Northwest',  # Kern
    
    # group 3: southeast counties
    '06059': 'Southeast',  # Orange
    '06073': 'Southeast',  # San Diego
    '06025': 'Southeast',  # Imperial
    '06065': 'Southeast',  # Riverside
    '06071': 'Southeast'   # San Bernardino
}

# list of counties to keep
county_fips_to_keep = list(county_groups.keys())

# filter counties dataset
counties_subset = counties[counties['fips'].isin(county_fips_to_keep)]

# spatial join to get FIPS codes in station data
counties_subset = counties_subset.to_crs(socal_data.crs)
plot_df = gpd.sjoin(socal_data, counties_subset[['fips', 'geometry']], 
                                 how='left', predicate='intersects')

# assign groups based on FIPS codes
plot_df['group'] = plot_df['fips'].map(county_groups)
plot_df = plot_df.drop(columns=['index_right'], errors='ignore')
plot_df

In [ ]:
# time series plot with same color per region
def plot_timeseries_same_color_per_region(plot_df, first_week):
    """
    Time series plot colored by region
    """
    # filter to first week and remove na values
    first_week_data = plot_df[
        plot_df['date'].isin(first_week)
    ].dropna(subset=['smokePM'])
    
    plt.figure(figsize=(14, 8))
    
    # colormap
    region_colors = {
        'Northwest': '#000000',
        'LA': '#888888',
        'Southeast': '#C5C5C5'
    }
    
    # pull unique groups
    unique_groups = first_week_data['group'].unique()
    
    for group in unique_groups:
        group_df = first_week_data[first_week_data['group'] == group]
        unique_groups = group_df['stn_id'].unique()
        
        for i, station in enumerate(unique_stations):
            station_data = group_df[group_df['stn_id'] == station].sort_values('date')
            
            # use same color for all stations in this region
            plt.plot(station_data['date'], station_data['smokePM'], 
                    color=region_colors[group],
                    marker='o', 
                    linewidth=2, 
                    markersize=4,
                    alpha=0.8,
                    label=group if i == 0 else "")  # label once per region
    
    plt.title('PM$_{2.5}$ by station and region - January 7-13, 2025', 
              fontsize=18, pad=20)
    plt.ylabel('PM2.5 (μg/m³)', fontsize=16)
    
    plt.xticks(first_week, [date.strftime('%m/%d') for date in first_week], rotation=45)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(title='Group', fontsize=14, title_fontsize=12)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.ylim(bottom=0)
    plt.show()

plot_timeseries_same_color_per_region(plot_df, first_week)


In [ ]:
def create_station_map_with_basemap(plot_df):
    """
    a map with stations colored by group on a contextily basemap
    """
        
    fig, ax = plt.subplots(figsize=(15, 12))
    
    # colormap
    region_colors = {
        'Northwest': '#000000',
        'LA': '#888888',
        'Southeast': '#C5C5C5'
    }
    
    for group in plot_df['group'].unique():
        group_df = plot_df[plot_df['group'] == group]
        group_df.plot(ax=ax, 
                        color=region_colors[group], 
                        markersize=75,
                        alpha=1,
                        edgecolor=region_colors[group],
                        linewidth=.5,
                        label=group)
    
    ctx.add_basemap(ax, 
                    crs=group_df.crs, 
                    source=ctx.providers.CartoDB.Voyager, 
                    zoom=10)
    
    ax.legend(title='Group', fontsize=12, title_fontsize=12, 
             loc='upper right', frameon=True, fancybox=True, shadow=True)
    
    ax.set_xticks([])
    ax.set_yticks([])
    
    plt.tight_layout()
    plt.show()

create_station_map_with_basemap(plot_df)

In [ ]:
def create_combined_plot(plot_df, first_week, save_path=None):
    """
    Side by side plot: map on left, time series on right
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    plt.subplots_adjust(hspace=0, wspace=0.1)
    
    # color mapping for regions
    region_colors = {
        'Northwest': '#000000',
        'LA': '#888888', 
        'Southeast': '#C5C5C5'
    }
    
    # define region order for consistent legend
    region_order = ['Northwest', 'LA', 'Southeast']
    
    # LEFT PLOT: map
    for group in region_order:
        if group in plot_df['group'].values:
            group_df = plot_df[plot_df['group'] == group]
            group_df.plot(ax=ax1,
                         color=region_colors[group],
                         markersize=125,
                         alpha=1,
                         edgecolor=region_colors[group],
                         linewidth=0.5,
                         label=group)
    
    ctx.add_basemap(ax1,
                   crs=plot_df.crs,
                   source=ctx.providers.CartoDB.Voyager,
                   zoom=10)
    
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    
    # RIGHT PLOT: time series
    # filter to first week and remove NA values
    first_week_data = plot_df[
        plot_df['date'].isin(first_week)
    ].dropna(subset=['smokePM'])
    
    # plot each station grouped by region in specified order
    for group in region_order:
        if group in first_week_data['group'].values:
            group_df = first_week_data[first_week_data['group'] == group]
            unique_stations = group_df['stn_id'].unique()
            
            for i, station in enumerate(unique_stations):
                station_data = group_df[group_df['stn_id'] == station].sort_values('date')
                
                ax2.plot(station_data['date'], station_data['smokePM'],
                        color=region_colors[group],
                        marker='o',
                        linewidth=2,
                        markersize=4,
                        alpha=0.8,
                        label=group if i == 0 else "")  # label once per region
    
    fig.suptitle('PM$_{2.5}$ by station and group - January 7-13, 2025', 
                fontsize=20, y=0.98)
    ax2.set_ylabel('PM$_{2.5}$ (μg/m³)', fontsize=14)
    ax2.set_xticks(first_week)
    ax2.set_xticklabels([date.strftime('%m/%d') for date in first_week], fontsize=12)
    ax2.tick_params(axis='y', labelsize=12)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['left'].set_visible(False)

    ax2.set_ylim(bottom=0)
    
    ax2.legend(title='Group', fontsize=14, title_fontsize=14)
    
    plt.tight_layout()
    
    # save
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")
    
    plt.show()

create_combined_plot(plot_df, first_week, save_path='../03_output/station_map_time_series.png')